In [73]:
import os
import xml.etree.ElementTree as ET
from datetime import datetime
from pymongo import MongoClient
import json
from typing import Dict, Any, Union, List
from dotenv import load_dotenv
# Langchain load data
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
### LLMs - Langchain
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
### Summary chain
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain, MapReduceChain, load_summarize_chain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_community.document_loaders import TextLoader

In [74]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_all_ids(collection):
    # Retrieve all _id values
    ids = collection.find({}, {"_id": 1})
    
    # Extract _id values into a list
    id_list = [doc["_id"] for doc in ids]
    
    return id_list
def get_ids_with_privacy_compliance(collection):
    # Query for documents that contain the "privacy-compliance" field
    docs = collection.find({"privacy-compliance": {"$exists": True}}, {"_id": 1})
    
    # Extract and return the list of _id values
    return [doc["_id"] for doc in docs]
def remove_common_elements(list_1, list_2):
    return [item for item in list_1 if item not in list_2]
def get_element_by_id(collection, id_value):
    # Find the document with the given _id
    result = collection.find_one({"_id": id_value})
    return result
def create_third_party_prompt(app_id,
                              app_tag_meta_data,
                              app_tag_action,
                              app_tag_application,
                              app_tag_data,
                              app_tag_property,
                              app_tag_provider,
                              app_tag_receiver,
                              app_tag_service,
                              app_tag_uses_configuration,
                              app_tag_uses_feature,
                              app_tag_uses_library):
    
    template = """
The app with the package name `{app_id}` is configured with the following manifest tags:
- Tag meta-data with attributes: {app_tag_meta_data}
- Tag action with attributes: {app_tag_action}
- Tag application with attributes: {app_tag_application}
- Tag data with attributes: {app_tag_data}
- Tag property with attributes: {app_tag_property}
- Tag provider with attributes: {app_tag_provider}
- Tag receiver with attributes: {app_tag_receiver}
- Tag service with attributes: {app_tag_service}
- Tag uses_configuration with attributes: {app_tag_uses_configuration}
- Tag uses_feature with attributes: {app_tag_uses_feature}
- Tag uses_library with attributes: {app_tag_uses_library}

Based on the above configuration, identify:
1. The third-party services that the app integrates with.
2. A list of corresponding import statements for each third-party service.

Respond in JSON format using one of the following examples:
- Case 1: suppose the app integrates with Google Firebase and Facebook
{{
    "Google Firebase": [
        "import com.google.firebase.messaging.FirebaseMessagingService",
        "import com.google.firebase.messaging.RemoteMessage"
    ],
    "Facebook": [
        "import com.facebook.FacebookSdk",
        "import com.facebook.appevents.AppEventsLogger"
    ]
}}

- Case 2: suppose the app does not integrate with any third-party service
{{
    "keyword": []
}}
Response with JSON structure only, don't add any explanation.
"""
    prompt = PromptTemplate.from_template(template)

    return prompt.format(
        app_id=app_id,
        app_tag_meta_data=app_tag_meta_data,
        app_tag_action=app_tag_action,
        app_tag_application=app_tag_application,
        app_tag_data=app_tag_data,
        app_tag_property=app_tag_property,
        app_tag_provider=app_tag_provider,
        app_tag_receiver=app_tag_receiver,
        app_tag_service=app_tag_service,
        app_tag_uses_configuration=app_tag_uses_configuration,
        app_tag_uses_feature=app_tag_uses_feature,
        app_tag_uses_library=app_tag_uses_library
    )
def extract_output_text_as_dict(response):
    """
    Extracts the 'output_text' from the response, cleans it, and converts it into a dictionary.

    :param response: The response dictionary containing 'output_text'.
    :return: A dictionary parsed from the 'output_text', or an empty dict if parsing fails.
    """
    if not response or not hasattr(response, "content"):  # Ensure response is valid
        print("Error: Invalid response object.")
        return {}

    output_text = response.content  # Assuming response.content holds the JSON string

    # Remove code block markers (```json and ```) if present
    if output_text.startswith("```json"):
        output_text = output_text[len("```json"):].strip("`").strip()

    try:
        # Convert the cleaned JSON string into a dictionary
        output_dict = json.loads(output_text)
        return output_dict
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return {}  # Return an empty dictionary instead of None
def insert_third_party_data(collection, condition_value: str, data: Dict[str, list]):
    """
    Insert or update the given dictionary data into MongoDB under the field 'keyword' using _id as condition.

    Args:
        collection (pymongo.collection.Collection): The MongoDB collection to interact with.
        condition_value (str): The _id value to use as a condition for insertion.
        data (Dict[str, list]): The dictionary data to insert or update.

    Returns:
        pymongo.results.UpdateResult: The result of the MongoDB update operation.
    """
    filter_query = {"_id": condition_value}
    update_query = {"$set": {"third-party-keyword": data}}
    
    # Insert or update the document using upsert=True
    result = collection.update_one(filter_query, update_query, upsert=True)
    return result

In [75]:
directory_path = r"C:\Users\ASUS\anaconda3-project-code\wearable-apk-manifest\manifest-wearable"
log_file = "analyzed-manifest-project-2.log"

In [76]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database_1 = 'wearable-project' 
mongoDB_collection_1 = 'wearable-app-02-04-2025'
# mongoDB_database_2 = 'wearable-project-2' 
# mongoDB_collection_2 = 'wearable-app'

In [77]:
collection_1 = connect_mongodb(mongoDB_uri, mongoDB_database_1, mongoDB_collection_1)
all_app_id = get_all_ids(collection_1)
print(len(all_app_id))

5162


In [78]:
analyzed_app_id = get_ids_with_privacy_compliance(collection_1)
print(len(analyzed_app_id))

641


In [79]:
final_app_id = remove_common_elements(all_app_id, analyzed_app_id)
print(len(final_app_id))

4521


In [80]:
#. Chat model
# Setup model
# Load environment variables
load_dotenv()

# Retrieve API key
api_key = os.getenv("OPENAI_API_KEY")
# Ensure the API key is correctly set
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

# Initialize the ChatOpenAI model
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=api_key  # Ensure you explicitly pass the API key
)

In [81]:
for i in range(2805,len(final_app_id)):
    print("----------------------- Loop "+str(i)+" -----------------------")
    app_id = final_app_id[i]
    print("App id: ", app_id)
    app_data = get_element_by_id(collection_1,app_id)
    # app_tag_meta_data = app_data["tag-meta-data"]
    # app_tag_action = app_data["tag-action"]
    # app_tag_application = app_data["tag-application"]
    # app_tag_data = app_data["tag-data"]
    # app_tag_property = app_data["tag-property"]
    # app_tag_provider = app_data["tag-provider"]
    # app_tag_receiver = app_data["tag-receiver"]
    # app_tag_service = app_data["tag-service"]
    # app_tag_uses_configuration = app_data["tag-uses-configuration"]
    # app_tag_uses_feature = app_data["tag-uses-feature"]
    # app_tag_uses_library = app_data["tag-uses-library"]
    app_tag_meta_data = app_data.get("tag-meta-data", "not found")
    app_tag_action = app_data.get("tag-action", "not found")
    app_tag_application = app_data.get("tag-application", "not found")
    app_tag_data = app_data.get("tag-data", "not found")
    app_tag_property = app_data.get("tag-property", "not found")
    app_tag_provider = app_data.get("tag-provider", "not found")
    app_tag_receiver = app_data.get("tag-receiver", "not found")
    app_tag_service = app_data.get("tag-service", "not found")
    app_tag_uses_configuration = app_data.get("tag-uses-configuration", "not found")
    app_tag_uses_feature = app_data.get("tag-uses-feature", "not found")
    app_tag_uses_library = app_data.get("tag-uses-library", "not found")
    prompt_template = create_third_party_prompt(
            app_id,
            app_tag_meta_data,
            app_tag_action,
            app_tag_application,
            app_tag_data,
            app_tag_property,
            app_tag_provider,
            app_tag_receiver,
            app_tag_service,
            app_tag_uses_configuration,
            app_tag_uses_feature,
            app_tag_uses_library
    )
    # print(prompt_template)
    response = llm.invoke(prompt_template)
    # print(response.content)
    output_dict = extract_output_text_as_dict(response)
    # print(output_dict)
    insert_status = insert_third_party_data(collection_1,app_id,output_dict)
    print("insert_status: ", insert_status)
    # break

----------------------- Loop 2805 -----------------------
App id:  com.wattbike.powerapp
insert_status:  UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)
----------------------- Loop 2806 -----------------------
App id:  com.drink.water.reminder.tracker
insert_status:  UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)
----------------------- Loop 2807 -----------------------
App id:  com.atletiq.app
insert_status:  UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)
----------------------- Loop 2808 -----------------------
App id:  jp.beautywalk.android
insert_status:  UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)
----------------------- Loop 2809 -----------------------
App id:  stoameditation.stoa
insert_status:  UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)
--